# PathSnap — Lightgbm
Standalone: shared setup (Cells 1–3) + this model.
Inputs under `/kaggle/input`: `train_features.parquet`, `student_test_features.parquet`, `train_alignment_diagnostics.parquet`, `student_test_alignment_diagnostics.parquet`. Raw trip folders are optional (enable route-continuity metrics).

### Cell 1: Config & Imports

In [ ]:
import os, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, roc_auc_score, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

# ---- Paths (Kaggle). Leave the parquet paths as None to auto-discover under INPUT_ROOT.
INPUT_ROOT = Path(os.environ.get("PATHSNAP_INPUT", "/kaggle/input"))
OUT_DIR    = Path(os.environ.get("PATHSNAP_OUT", "/kaggle/working"))
TRAIN_PARQUET        = None   # e.g. INPUT_ROOT / "pathsnap-features" / "train_features.parquet"
STUDENT_TEST_PARQUET = None   # e.g. INPUT_ROOT / "pathsnap-features" / "student_test_features.parquet"

MODEL_DIR   = OUT_DIR / "models";  MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR  = OUT_DIR / "results"; RESULT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR    = OUT_DIR / "data";    DATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "dist_to_edge_m", "dist_to_u_node_m", "dist_to_v_node_m",
    "bearing_diff_deg", "bearing_valid",
    "time_gap_prev_s", "speed_implied_mps", "irregular_gap_flag",
    "candidate_rank", "dist_minus_best_at_point_m", "num_candidates_at_point",
    "edge_length_m", "u_node_degree", "v_node_degree", "reverse_edge_exists",
]
TARGET = "label_point"        # the parquet `label` (point-level, NaN = unusable point), renamed in Cell 2

N_FOLDS = 5                   # GroupKFold on trip_id, 80 training trips

def tkey(t):
    """Normalise trip ids so '1', 1 and '00000001' all match folder names."""
    s = str(t).strip()
    return s.zfill(8) if s.isdigit() else s

def find_one(name):
    hits = sorted(INPUT_ROOT.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}")
    if len(hits) > 1:
        print(f"[warn] multiple {name} found, using {hits[0]}")
    return hits[0]

TRAIN_PARQUET = Path(TRAIN_PARQUET) if TRAIN_PARQUET else find_one("train_features.parquet")
STUDENT_TEST_PARQUET = Path(STUDENT_TEST_PARQUET) if STUDENT_TEST_PARQUET else find_one("student_test_features.parquet")

# Optional: raw trip folders (any folder holding a *.route file) -> enables continuity metrics.
TRIP_DIRS = {tkey(p.parent.name): p.parent for p in INPUT_ROOT.rglob("*.route")}
print(f"train parquet: {TRAIN_PARQUET}\nstudent-test parquet: {STUDENT_TEST_PARQUET}")
print(f"raw labelled trip folders found: {len(TRIP_DIRS)}")


### Cell 2: Load Features + Point-Level Labels (From The Parsing Pipeline)

In [ ]:
# The parsing notebook already aligns each GPS point to the ordered .route:
#   * `label` is point-level: exactly 1 positive candidate per usable point, NaN for points
#     that are not usable (reference arc missing from candidates, or uncertain alignment).
#   * *_alignment_diagnostics.parquet gives the reference arc per point (aligned_edge_id).
# Raw trip folders are OPTIONAL here: if found, they enable route-continuity metrics.

def find_optional(name):
    hits = sorted(INPUT_ROOT.rglob(name))
    return hits[0] if hits else None

TRAIN_ALIGN = find_one("train_alignment_diagnostics.parquet")
TEST_ALIGN  = find_one("student_test_alignment_diagnostics.parquet")

R_EARTH = 6371008.8
TRIP_INFO = {}   # trip_key -> dict(arc_u, arc_v, route); filled only if raw trip folders exist

def _read_cols(path, ncols, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None, usecols=list(range(ncols)),
                       engine="c").to_numpy(dtype=dtype)

def load_graph(tk):
    d = TRIP_DIRS[tk]
    arcs = _read_cols(next(d.glob("*.arcs")), 2, dtype=np.int64)
    route = _read_cols(next(d.glob("*.route")), 1, dtype=np.int64).ravel()
    TRIP_INFO[tk] = {"arc_u": arcs[:, 0].astype(np.int32), "arc_v": arcs[:, 1].astype(np.int32),
                     "route": route.astype(np.int32)}

def prepare(feat_path, align_path, partition_name):
    df = pd.read_parquet(feat_path)
    al = pd.read_parquet(align_path)
    df["trip_key"] = df["trip_id"].map(tkey)
    al["trip_key"] = al["trip_id"].map(tkey)
    df = df.rename(columns={"label": TARGET})
    al = al[["trip_key", "point_idx", "aligned_edge_id", "candidate_hit",
             "alignment_status", "alignment_distance_m", "usable_for_training"]]
    al["point_idx"] = al["point_idx"].astype(np.int64)
    df = df.merge(al, on=["trip_key", "point_idx"], how="left", validate="many_to_one")
    assert df["aligned_edge_id"].notna().all(), "some points have no alignment row"
    df["ref_arc"] = df["aligned_edge_id"].astype(np.int64)
    df["ref_in_candidates"] = df["candidate_hit"].astype(np.int8)
    df["ref_known"] = (df["alignment_status"] == "accepted").astype(np.int8)
    df = df.sort_values(["trip_key", "point_idx", "candidate_rank"]).reset_index(drop=True)

    # Sanity: on labelled rows, label == (candidate is the reference arc)
    lab = df[TARGET].notna()
    agree = (df.loc[lab, TARGET].astype(int) == (df.loc[lab, "candidate_edge_id"] == df.loc[lab, "ref_arc"]).astype(int)).mean()
    per_pt = df[lab].groupby(["trip_key", "point_idx"])[TARGET].sum()
    pts = df.drop_duplicates(["trip_key", "point_idx"])
    print(f"[{partition_name}] trips={df['trip_key'].nunique()} rows={len(df):,} points={len(pts):,} | "
          f"labelled rows={lab.mean():.1%} | label==ref agreement={agree:.4f} | "
          f"positives per labelled point={per_pt.mean():.3f} | candidate miss={1 - pts['candidate_hit'].mean():.2%} | "
          f"uncertain alignments={(pts['alignment_status'] != 'accepted').mean():.2%}")

    for tk in df["trip_key"].unique():
        if tk in TRIP_DIRS:
            load_graph(tk)
    return df

train_df = prepare(TRAIN_PARQUET, TRAIN_ALIGN, "train")
test_df  = prepare(STUDENT_TEST_PARQUET, TEST_ALIGN, "student_test")
assert not set(train_df["trip_key"]) & set(test_df["trip_key"]), "trip overlap between train and student-test!"
print(f"raw graphs loaded for continuity: {len(TRIP_INFO)} trips "
      f"({'continuity metrics ON' if TRIP_INFO else 'continuity metrics OFF - attach raw trip folders to enable'})")

train_fit_df = train_df                                   # all rows (unlabelled rows are still scored for Top-1)
X_train = train_fit_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
y_train_raw = train_fit_df[TARGET].to_numpy()
LAB_TRAIN = ~np.isnan(y_train_raw)                        # rows usable for fitting
y_train = np.where(LAB_TRAIN, y_train_raw, 0).astype(int)  # NaN rows never used for fitting/metrics
groups  = train_fit_df["trip_key"].to_numpy()
X_test  = test_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)


### Cell 3: Metrics + Experiment Runner

In [ ]:
def _out_lookup(tk):
    info = TRIP_INFO[tk]
    if "u_order" not in info:
        order = np.argsort(info["arc_u"], kind="stable")
        info["u_order"], info["u_sorted"] = order, info["arc_u"][order]
    return info

def continuity(tk, seq):
    """Fraction of consecutive point transitions that are physically possible.
    strict : same arc, or prev.v == next.u (1 hop)
    relaxed: strict, or reachable through one intermediate arc (2 hops)"""
    if len(seq) < 2:
        return np.nan, np.nan, 0
    info = _out_lookup(tk)
    u, v = info["arc_u"], info["arc_v"]
    a, b = seq[:-1], seq[1:]
    strict = (a == b) | (v[a] == u[b])
    relaxed = strict.copy()
    for k in np.where(~strict)[0]:
        n = v[a[k]]
        lo, hi = np.searchsorted(info["u_sorted"], [n, n + 1])
        mids = info["u_order"][lo:hi]
        relaxed[k] = np.any(v[mids] == u[b[k]])
    return strict.mean(), relaxed.mean(), len(a)

def evaluate(df, proba, threshold=0.5):
    lab = df[TARGET].notna().to_numpy()
    y = df[TARGET].to_numpy()[lab].astype(int)
    pl = proba[lab]
    pred = (pl >= threshold).astype(int)
    pc = np.clip(pl, 1e-7, 1 - 1e-7)
    m = {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "log_loss": log_loss(y, pc, labels=[0, 1]),
        "roc_auc": roc_auc_score(y, pl) if len(np.unique(y)) > 1 else np.nan,
        "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
    }
    d = df[["trip_key", "point_idx", "candidate_edge_id", "ref_arc", "ref_in_candidates", "ref_known"]].copy()
    d["p"] = proba
    top = (d.sort_values(["trip_key", "point_idx", "p"], ascending=[True, True, False])
             .drop_duplicates(["trip_key", "point_idx"]))
    known = top["ref_known"].to_numpy() == 1                            # points with an accepted reference arc
    correct = (top["candidate_edge_id"] == top["ref_arc"]).to_numpy()
    covered = top["ref_in_candidates"].to_numpy() == 1
    m["top1_accuracy"] = correct[known].mean()                          # candidate misses count as wrong
    m["top1_accuracy_covered"] = correct[known & covered].mean()
    m["candidate_miss_rate"] = 1 - covered[known].mean()
    s_num = r_num = n_tr = 0.0
    inter = n_pred = n_true = 0
    for tk, g in top.groupby("trip_key", sort=False):
        if tk not in TRIP_INFO:
            continue
        seq = g["candidate_edge_id"].to_numpy()
        s, r, n = continuity(tk, seq)
        if n:
            s_num += s * n; r_num += r * n; n_tr += n
        ps, ts = set(seq.tolist()), set(TRIP_INFO[tk]["route"].tolist())
        inter += len(ps & ts); n_pred += len(ps); n_true += len(ts)
    m["route_continuity_strict"] = s_num / n_tr if n_tr else np.nan
    m["route_continuity_relaxed"] = r_num / n_tr if n_tr else np.nan
    m["route_arc_precision"] = inter / n_pred if n_pred else np.nan
    m["route_arc_recall"] = inter / n_true if n_true else np.nan
    return m

def make_pipeline(estimator):
    # Same preprocessing for every model (fit inside each CV fold -> no leakage).
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", estimator)])

def _subsample(idx, max_rows, rng):
    if max_rows is None or len(idx) <= max_rows:
        return idx
    return np.sort(rng.choice(idx, size=max_rows, replace=False))

def _jsonable(x):
    if isinstance(x, dict):
        return {k: _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    return str(x)

def run_experiment(name, family, estimator, key_config, max_train_rows=None, n_folds=N_FOLDS):
    rng = np.random.default_rng(SEED)
    pipe = make_pipeline(estimator)
    print(f"\n==================== {name} ({family}) ====================")

    # ---- 1) Group-aware CV on the training trips
    fold_rows = []
    oof = np.full(len(y_train), np.nan)
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=n_folds).split(X_train, y_train, groups)):
        tr = _subsample(tr[LAB_TRAIN[tr]], max_train_rows, rng)
        m = clone(pipe)
        t0 = time.perf_counter(); m.fit(X_train[tr], y_train[tr]); fit_s = time.perf_counter() - t0
        t0 = time.perf_counter(); p = m.predict_proba(X_train[va])[:, 1]; pred_s = time.perf_counter() - t0
        oof[va] = p
        met = evaluate(train_fit_df.iloc[va], p)
        met.pop("confusion_matrix")
        fold_rows.append({"fold": fold, "fit_s": fit_s, "predict_s": pred_s, **met})
        print(f"  fold {fold}: macroF1={met['macro_f1']:.4f} top1={met['top1_accuracy']:.4f} "
              f"logloss={met['log_loss']:.4f} cont={met['route_continuity_strict']:.4f} fit={fit_s:.1f}s")
    folds = pd.DataFrame(fold_rows)
    folds.to_csv(RESULT_DIR / f"{name}_cv_folds.csv", index=False)
    oof_df = train_fit_df[["trip_key", "point_idx", "candidate_edge_id"]].copy()
    oof_df["proba"] = oof
    oof_df.to_parquet(RESULT_DIR / f"{name}_oof_train.parquet", index=False)   # for HMM tuning later

    # ---- 2) Refit on all training trips, evaluate on the 10 student-testing trips
    full_idx = _subsample(np.where(LAB_TRAIN)[0], max_train_rows, rng)
    final = clone(pipe)
    t0 = time.perf_counter(); final.fit(X_train[full_idx], y_train[full_idx]); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); p_test = final.predict_proba(X_test)[:, 1]; infer_s = time.perf_counter() - t0
    tm = evaluate(test_df, p_test)

    # Train-set score on the refit model (for over/underfitting evidence)
    lab_idx = np.where(LAB_TRAIN)[0]
    tr_eval_idx = rng.choice(lab_idx, size=min(len(lab_idx), 200_000), replace=False)
    p_tr = final.predict_proba(X_train[tr_eval_idx])[:, 1]
    tr_ll = log_loss(y_train[tr_eval_idx], np.clip(p_tr, 1e-7, 1 - 1e-7), labels=[0, 1])
    tr_f1 = f1_score(y_train[tr_eval_idx], (p_tr >= 0.5).astype(int), average="macro")

    pred_df = test_df[["trip_key", "point_idx", "candidate_edge_id"]].copy()
    pred_df["proba"] = p_test
    pred_df.to_parquet(RESULT_DIR / f"{name}_student_test_proba.parquet", index=False)
    joblib.dump(final, MODEL_DIR / f"{name}.joblib", compress=3)

    # Confusion matrix figure (student-test)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ConfusionMatrixDisplay(np.array(tm["confusion_matrix"]), display_labels=["not ref", "ref arc"]).plot(
        ax=ax, colorbar=False, values_format="d")
    ax.set_title(f"{name} — student-test")
    fig.tight_layout(); fig.savefig(RESULT_DIR / f"{name}_confusion_matrix.png", dpi=150); plt.show()

    summary = {
        "model": name, "family": family, "key_config": key_config,
        "cv_macro_f1_mean": folds["macro_f1"].mean(), "cv_macro_f1_std": folds["macro_f1"].std(),
        "cv_top1_mean": folds["top1_accuracy"].mean(), "cv_top1_std": folds["top1_accuracy"].std(),
        "cv_log_loss_mean": folds["log_loss"].mean(),
        "cv_continuity_mean": folds["route_continuity_strict"].mean(),
        "train_macro_f1": tr_f1, "train_log_loss": tr_ll,
        **{f"st_{k}": v for k, v in tm.items() if k != "confusion_matrix"},
        "train_time_s": train_s,
        "inference_ms_per_1k_rows": 1000 * infer_s / len(X_test) * 1000,
        "n_train_rows_used": int(len(full_idx)),
    }
    record = {"summary": summary, "seed": SEED,
              "params": _jsonable(estimator.get_params()),
              "student_test_confusion_matrix": tm["confusion_matrix"],
              "cv_folds": _jsonable(fold_rows)}
    with open(RESULT_DIR / f"{name}_results.json", "w") as f:
        json.dump(_jsonable(record), f, indent=2)

    comp_path = RESULT_DIR / "model_comparison.csv"
    comp = pd.read_csv(comp_path) if comp_path.exists() else pd.DataFrame()
    if len(comp):
        comp = comp[comp["model"] != name]
    pd.concat([comp, pd.DataFrame([summary])], ignore_index=True).to_csv(comp_path, index=False)

    print(f"  CV  macroF1={summary['cv_macro_f1_mean']:.4f}±{summary['cv_macro_f1_std']:.4f} "
          f"top1={summary['cv_top1_mean']:.4f}±{summary['cv_top1_std']:.4f}")
    print(f"  TRAIN macroF1={tr_f1:.4f} logloss={tr_ll:.4f}")
    print(f"  ST  macroF1={tm['macro_f1']:.4f} top1={tm['top1_accuracy']:.4f} "
          f"(covered {tm['top1_accuracy_covered']:.4f}) logloss={tm['log_loss']:.4f} "
          f"continuity strict/relaxed={tm['route_continuity_strict']:.4f}/{tm['route_continuity_relaxed']:.4f}")
    print(f"  train {train_s:.1f}s | inference {summary['inference_ms_per_1k_rows']:.2f} ms / 1k rows")
    return final, summary


### Cell 6: Lightgbm

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    objective="binary", random_state=SEED, n_jobs=-1, verbose=-1)
lgbm_model, lgbm_summary = run_experiment(
    "lightgbm", "Boosting", lgbm,
    key_config="600 trees, lr=0.05, num_leaves=63, min_child_samples=50, bagging 0.8/0.8")
